In [1]:
# ==============================================================================
# 12. Model Training (XGBoost)
# ==============================================================================
#
# 목적:
#   1. XGBoost v2 모델 학습
#   2. Validation 성능 평가
#   3. Test 성능 평가
#   4. 모델 저장
#
# 입력: train.csv, val.csv, test.csv, scale_pos_weights.json
# 출력: 학습된 모델 (.pkl), 성능 리포트
# ==============================================================================

import pandas as pd
import numpy as np
import json
import os
import pickle
from datetime import datetime

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, 
    precision_recall_curve, 
    auc, 
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
import warnings
warnings.filterwarnings('ignore')

# 피처 설정 (top33, top21, top10, top5)
FEATURE_CONFIG = 'top21'

INPUT_DIR = f'../data/processed/{FEATURE_CONFIG}'
OUTPUT_DIR = f'../models/{FEATURE_CONFIG}'

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"=== 12. Model Training 시작 ({FEATURE_CONFIG}) ===")

# --- 데이터 로드 ---
print("\nStep 1: 데이터 로드")

df_train = pd.read_csv(os.path.join(INPUT_DIR, 'train.csv'))
df_val = pd.read_csv(os.path.join(INPUT_DIR, 'val.csv'))
df_test = pd.read_csv(os.path.join(INPUT_DIR, 'test.csv'))

print(f"✓ Train: {len(df_train):,} rows")
print(f"✓ Val: {len(df_val):,} rows")
print(f"✓ Test: {len(df_test):,} rows")

# --- 피처/레이블 로드 ---
with open(f'../data/processed/{FEATURE_CONFIG}_features.json', 'r') as f:
    feature_cols = json.load(f)

with open(os.path.join(INPUT_DIR, 'scale_pos_weights.json'), 'r') as f:
    scale_pos_weights = json.load(f)

label_cols = [col for col in df_train.columns if 'next_' in col]

print(f"✓ 피처: {len(feature_cols)}개")
print(f"✓ 레이블: {len(label_cols)}개")

=== 12. Model Training 시작 (top21) ===

Step 1: 데이터 로드
✓ Train: 657,172 rows
✓ Val: 141,844 rows
✓ Test: 142,801 rows
✓ 피처: 21개
✓ 레이블: 12개


In [2]:
# ==============================================================================
# Step 2: 피처/레이블 분리
# ==============================================================================

print("\nStep 2: 피처/레이블 분리")

X_train = df_train[feature_cols]
X_val = df_val[feature_cols]
X_test = df_test[feature_cols]

print(f"✓ X_train shape: {X_train.shape}")
print(f"✓ X_val shape: {X_val.shape}")
print(f"✓ X_test shape: {X_test.shape}")

# 결측 확인
train_missing = X_train.isna().sum().sum()
val_missing = X_val.isna().sum().sum()
test_missing = X_test.isna().sum().sum()

if train_missing + val_missing + test_missing == 0:
    print("✓ 결측 없음 확인")
else:
    print(f"⚠️ 결측 발견: Train={train_missing}, Val={val_missing}, Test={test_missing}")


Step 2: 피처/레이블 분리
✓ X_train shape: (657172, 21)
✓ X_val shape: (141844, 21)
✓ X_test shape: (142801, 21)
✓ 결측 없음 확인


In [3]:
# ==============================================================================
# Step 3: 학습 타겟 선택
# ==============================================================================
#
# 주요 타겟:
#   - death_next_24h: 24시간 내 사망 예측
#   - vent_next_24h: 24시간 내 인공호흡기 시작 예측
#   - pressor_next_24h: 24시간 내 승압제 시작 예측
#   - composite_next_24h: 위 3개 중 하나라도 발생
#
# 여기서는 모든 레이블에 대해 학습하되, 메인 타겟 지정
# ==============================================================================

print("\nStep 3: 학습 타겟 선택")

# 메인 타겟 (우선순위 높은 것들)
main_targets = [
    'death_next_24h',
    'vent_next_24h', 
    'pressor_next_24h',
    'composite_next_24h'
]

# 존재하는 타겟만 필터링
main_targets = [t for t in main_targets if t in label_cols]

print(f"메인 타겟: {main_targets}")


Step 3: 학습 타겟 선택
메인 타겟: ['death_next_24h', 'vent_next_24h', 'pressor_next_24h', 'composite_next_24h']


In [4]:
# ==============================================================================
# Step 4: 모델 학습 함수 정의 (v2 - Threshold 분석 추가)
# ==============================================================================

def train_and_evaluate(model, model_name, X_train, y_train, X_val, y_val, X_test, y_test, target_name):
    """
    모델 학습 및 평가 (v2: Threshold별 성능 분석 추가)
    """
    print(f"\n{'='*50}")
    print(f"{model_name} - {target_name}")
    print('='*50)
    
    # 학습
    print("학습 중...")
    
    if model_name == 'XGBoost':
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )
    else:  # LightGBM
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )
    
    # 예측
    y_train_pred = model.predict_proba(X_train)[:, 1]
    y_val_pred = model.predict_proba(X_val)[:, 1]
    y_test_pred = model.predict_proba(X_test)[:, 1]
    
    # 평가 지표 계산
    results = {
        'model': model_name,
        'target': target_name,
        'train_auroc': roc_auc_score(y_train, y_train_pred),
        'val_auroc': roc_auc_score(y_val, y_val_pred),
        'test_auroc': roc_auc_score(y_test, y_test_pred),
    }
    
    # AUPRC
    precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_pred)
    results['test_auprc'] = auc(recall_curve, precision_curve)
    
    # 결과 출력
    print(f"\n  Train AUROC: {results['train_auroc']:.4f}")
    print(f"  Val AUROC:   {results['val_auroc']:.4f}")
    print(f"  Test AUROC:  {results['test_auroc']:.4f}")
    print(f"  Test AUPRC:  {results['test_auprc']:.4f}")
    
    # Overfitting 체크
    overfit_gap = results['train_auroc'] - results['val_auroc']
    if overfit_gap > 0.05:
        print(f"\n  ⚠️ Overfitting 의심: Train-Val gap = {overfit_gap:.4f}")
    else:
        print(f"\n  ✅ Overfitting 양호: Train-Val gap = {overfit_gap:.4f}")
    
    # ===========================================
    # Threshold별 성능 분석 (추가)
    # ===========================================
    print(f"\n  === Threshold별 성능 ===")
    print(f"  {'Threshold':<10} {'Recall':<10} {'Precision':<10} {'F1':<10}")
    print(f"  {'-'*40}")
    
    threshold_results = []
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.85]:
        y_pred_binary = (y_test_pred >= threshold).astype(int)
        rec = recall_score(y_test, y_pred_binary, zero_division=0)
        prec = precision_score(y_test, y_pred_binary, zero_division=0)
        f1 = f1_score(y_test, y_pred_binary, zero_division=0)
        
        threshold_results.append({
            'threshold': threshold,
            'recall': rec,
            'precision': prec,
            'f1': f1
        })
        
        print(f"  {threshold:<10.2f} {rec:<10.3f} {prec:<10.3f} {f1:<10.3f}")
    
    # 임상 권장: Recall >= 0.7 중 F1 최대
    clinical_candidates = [t for t in threshold_results if t['recall'] >= 0.7]
    if clinical_candidates:
        best_clinical = max(clinical_candidates, key=lambda x: x['f1'])
        print(f"\n  📌 임상 권장 (Recall≥70%): threshold={best_clinical['threshold']:.2f}")
        print(f"     → Recall={best_clinical['recall']:.3f}, Precision={best_clinical['precision']:.3f}, F1={best_clinical['f1']:.3f}")
        results['clinical_threshold'] = best_clinical['threshold']
        results['clinical_recall'] = best_clinical['recall']
        results['clinical_precision'] = best_clinical['precision']
        results['clinical_f1'] = best_clinical['f1']
    else:
        print(f"\n  ⚠️ Recall≥70% 달성하는 threshold 없음")
        results['clinical_threshold'] = 0.5
    
    # F1 기준 best threshold
    best_f1_result = max(threshold_results, key=lambda x: x['f1'])
    results['best_threshold'] = best_f1_result['threshold']
    results['test_f1'] = best_f1_result['f1']
    
    print(f"\n  📊 F1 기준 best: threshold={best_f1_result['threshold']:.2f}, F1={best_f1_result['f1']:.3f}")
    
    # Confusion Matrix (임상 권장 threshold 기준)
    chosen_threshold = results.get('clinical_threshold', 0.5)
    y_pred_binary = (y_test_pred >= chosen_threshold).astype(int)
    cm = confusion_matrix(y_test, y_pred_binary)
    
    print(f"\n  Confusion Matrix (threshold={chosen_threshold:.2f}):")
    print(f"    TN={cm[0,0]:,}  FP={cm[0,1]:,}")
    print(f"    FN={cm[1,0]:,}  TP={cm[1,1]:,}")
    
    return results, model, y_test_pred

In [5]:
# ==============================================================================
# Step 5: XGBoost 학습
# ==============================================================================

print("\n" + "="*60)
print("Step 5: XGBoost 학습")
print("="*60)

xgb_results = []
xgb_models = {}
xgb_predictions = {}

for target in main_targets:
    # 레이블 추출
    y_train = df_train[target]
    y_val = df_val[target]
    y_test = df_test[target]
    
    # scale_pos_weight
    spw = scale_pos_weights.get(target, 1.0)
    
    # 모델 정의
    xgb_model = XGBClassifier(
        n_estimators=200,      # 줄이기
        max_depth=4,           # 6 → 4
        learning_rate=0.03,    # 더 천천히
        min_child_weight=10,   # 추가
        reg_alpha=0.1,         # L1 정규화
        reg_lambda=1.0,        # L2 정규화
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        random_state=42,
        n_jobs=-1,
        eval_metric='auc',
        early_stopping_rounds=30
    )
    
    # 학습 및 평가
    result, model, y_pred = train_and_evaluate(
        xgb_model, 'XGBoost', 
        X_train, y_train, 
        X_val, y_val, 
        X_test, y_test,
        target
    )
    
    xgb_results.append(result)
    xgb_models[target] = model
    xgb_predictions[target] = y_pred


Step 5: XGBoost 학습

XGBoost - death_next_24h
학습 중...

  Train AUROC: 0.9576
  Val AUROC:   0.8945
  Test AUROC:  0.9334
  Test AUPRC:  0.3144

  ⚠️ Overfitting 의심: Train-Val gap = 0.0631

  === Threshold별 성능 ===
  Threshold  Recall     Precision  F1        
  ----------------------------------------
  0.30       0.917      0.029      0.056     
  0.40       0.880      0.037      0.072     
  0.50       0.816      0.048      0.090     
  0.60       0.759      0.062      0.115     
  0.70       0.684      0.088      0.157     
  0.80       0.573      0.148      0.235     
  0.85       0.503      0.216      0.302     

  📌 임상 권장 (Recall≥70%): threshold=0.60
     → Recall=0.759, Precision=0.062, F1=0.115

  📊 F1 기준 best: threshold=0.85, F1=0.302

  Confusion Matrix (threshold=0.60):
    TN=129,440  FP=12,290
    FN=258  TP=813

XGBoost - vent_next_24h
학습 중...

  Train AUROC: 0.7806
  Val AUROC:   0.6933
  Test AUROC:  0.7160
  Test AUPRC:  0.0923

  ⚠️ Overfitting 의심: Train-Val gap = 0.08

In [6]:
# Step 6: LightGBM (생략 가능 - XGBoost가 더 좋음)

In [7]:
# ==============================================================================
# Step 7: 결과 비교
# ==============================================================================

print("\n" + "="*60)
print("Step 7: 결과 비교")
print("="*60)

# v1 결과 로드 (있으면)
v1_path = '../models/results_20260107_222324.csv'
if os.path.exists(v1_path):
    v1_results = pd.read_csv(v1_path)
else:
    print("v1 결과 없음, 비교 스킵")

# v2 결과
v2_results = pd.DataFrame(xgb_results)

# 비교
comparison = pd.merge(
    v1_results[v1_results['model']=='XGBoost'],
    v2_results,
    on='target',
    suffixes=('_v1', '_v2')
)

print("=== v1 vs v2 비교 ===")
print(comparison[['target', 'test_auroc_v1', 'test_auroc_v2', 'val_auroc_v1', 'val_auroc_v2']])


Step 7: 결과 비교
=== v1 vs v2 비교 ===
               target  test_auroc_v1  test_auroc_v2  val_auroc_v1  \
0      death_next_24h       0.927484       0.933404      0.882852   
1       vent_next_24h       0.753347       0.715950      0.740854   
2    pressor_next_24h       0.821893       0.796989      0.815285   
3  composite_next_24h       0.785876       0.755665      0.763441   

   val_auroc_v2  
0      0.894508  
1      0.693292  
2      0.800293  
3      0.742076  


In [8]:
# ==============================================================================
# Step 8: 모델 저장 (XGBoost only, v2)
# ==============================================================================

print("\n" + "="*60)
print("Step 8: 모델 저장 (XGBoost v2)")
print("="*60)

from datetime import datetime
import os
import pickle
import pandas as pd

# 타임스탬프
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 실험 버전 태그
EXP_TAG = f"v2_{FEATURE_CONFIG}"

# ==============================================================================
# XGBoost 모델 저장
# ==============================================================================
for target, model in xgb_models.items():
    model_path = os.path.join(
        OUTPUT_DIR,
        f'xgb_{target}_{EXP_TAG}_{timestamp}.pkl'
    )
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f"✓ {model_path}")

# ==============================================================================
# 결과 요약 저장
# ==============================================================================
results_path = os.path.join(
    OUTPUT_DIR,
    f'results_{EXP_TAG}_{timestamp}.csv'
)
v2_results.to_csv(results_path, index=False)
print(f"✓ {results_path}")

# ==============================================================================
# Test 예측값 저장 (SHAP 분석 / 후처리용)
# ==============================================================================
predictions = {
    'stay_id': df_test['stay_id'].values,
    'observation_hour': df_test['observation_hour'].values,
}

for target in main_targets:
    predictions[f'xgb_{target}_prob'] = xgb_predictions[target]
    predictions[f'{target}_actual'] = df_test[target].values

df_predictions = pd.DataFrame(predictions)

pred_path = os.path.join(
    OUTPUT_DIR,
    f'test_predictions_{EXP_TAG}_{timestamp}.csv'
)
df_predictions.to_csv(pred_path, index=False)
print(f"✓ {pred_path}")

print("\n=== Model Training v2 (XGBoost only) 완료 ===")


Step 8: 모델 저장 (XGBoost v2)
✓ ../models/top21/xgb_death_next_24h_v2_top21_20260109_115014.pkl
✓ ../models/top21/xgb_vent_next_24h_v2_top21_20260109_115014.pkl
✓ ../models/top21/xgb_pressor_next_24h_v2_top21_20260109_115014.pkl
✓ ../models/top21/xgb_composite_next_24h_v2_top21_20260109_115014.pkl
✓ ../models/top21/results_v2_top21_20260109_115014.csv
✓ ../models/top21/test_predictions_v2_top21_20260109_115014.csv

=== Model Training v2 (XGBoost only) 완료 ===
